last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [39]:
import os


In [40]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [41]:
# 직접 입력 방식 (비권장 — 공유 시 키 노출 위험)
# os.environ['OPENAI_API_KEY'] = "sk-..."

In [42]:
#! curl ipinfo.io

In [43]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.


### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [45]:
!pip install -q "pydantic>=2.0,<2.10"
!pip install -q "langchain-community>=0.3"

In [46]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

ValueError: File path /content/Demian.pdf is not a valid file or url

In [50]:
from google.colab import files
uploaded = files.upload()  # 파일 선택 창이 열립니다

KeyboardInterrupt: 

In [49]:
# Demian.pdf를 직접 다운로드
!wget -q "https://www.holybooks.com/wp-content/uploads/Demian-By-Hermann-Hesse.pdf" -O /content/Demian.pdf

# 확인
import os
if os.path.exists("/content/Demian.pdf"):
    print("✅ 다운로드 성공!")
else:
    print("❌ 실패")

✅ 다운로드 성공!


In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load()
print(f"✅ 총 페이지 수: {len(pages)}")
print(pages[0].page_content[:200])

/tmp/ipykernel_21687/1247963210.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ 총 페이지 수: 182
DEMIAN 
• 
Downloaded from https://www.holybooks.com


In [2]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [3]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [4]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [5]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

RuntimeError: Error loading /content/titanic.csv

In [6]:
# titanic.csv 직접 다운로드
!wget -q "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv" -O /content/titanic.csv

# 확인
import os
if os.path.exists("/content/titanic.csv"):
    print("✅ 다운로드 성공!")
else:
    print("❌ 실패")

✅ 다운로드 성공!


In [7]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")
data = loader.load()

print(f"✅ 총 행 수: {len(data)}")
print(f"\n--- 첫 번째 행 ---")
print(data[0].page_content)
print(f"\n--- 메타데이터 ---")
print(data[0].metadata)

✅ 총 행 수: 891

--- 첫 번째 행 ---
PassengerId: 1
Survived: 0
Pclass: 3
Name: Braund, Mr. Owen Harris
Sex: male
Age: 22
SibSp: 1
Parch: 0
Ticket: A/5 21171
Fare: 7.25
Cabin: 
Embarked: S

--- 메타데이터 ---
{'source': '/content/titanic.csv', 'row': 0}


In [8]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [9]:
from langchain_community.document_loaders import WebBaseLoader

In [10]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [11]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [17]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [13]:
# state_of_the_union.txt 다운로드
!wget -q "https://raw.githubusercontent.com/langchain-ai/langchain/master/docs/docs/modules/state_of_the_union.txt" -O /content/state_of_the_union.txt

# 확인
import os
if os.path.exists("/content/state_of_the_union.txt"):
    print("✅ 다운로드 성공!")
else:
    print("❌ 실패")

✅ 다운로드 성공!


In [14]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

print(f"✅ 텍스트 길이: {len(text)} 글자")
print(text[:300])

✅ 텍스트 길이: 0 글자



In [15]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [18]:
print(chunks[0])

IndexError: list index out of range

In [20]:
# 파일 읽기 + 분할 한 번에 실행
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.create_documents([text])

print(f"✅ 청크 수: {len(chunks)}")
print(chunks[0])

✅ 청크 수: 0


IndexError: list index out of range

In [22]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

from langchain_text_splitters import RecursiveCharacterTextSplitter

# separator 대신 RecursiveCharacterTextSplitter 사용
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.create_documents([text])

print(f"✅ 청크 수: {len(chunks)}")
print(chunks[0])

✅ 청크 수: 0


IndexError: list index out of range

In [23]:
# 파일 내용 확인
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

print(f"텍스트 길이: {len(text)}")
print(f"앞 200글자:\n{text[:200]}")

텍스트 길이: 0
앞 200글자:



In [26]:
# 파일 재다운로드
!wget -q "https://raw.githubusercontent.com/hwchase17/chroma-langchain/master/state_of_the_union.txt" -O /content/state_of_the_union.txt

with open("/content/state_of_the_union.txt") as f:
    text = f.read()

print(f"✅ 텍스트 길이: {len(text)} 글자")
print(text[:200])

✅ 텍스트 길이: 38539 글자
Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. 


In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.create_documents([text])
print(f"✅ 청크 수: {len(chunks)}")
print(chunks[0])

✅ 청크 수: 49
page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.'


각 chunk의 길이를 확인해보겠습니다,

In [29]:
length = []
for chunk in chunks:
    length.append(len(chunk.page_content))  # .page_content 추가

print(length)

[939, 916, 994, 874, 906, 923, 951, 967, 952, 930, 962, 956, 973, 974, 858, 973, 993, 958, 978, 956, 842, 889, 942, 975, 974, 892, 977, 958, 951, 924, 975, 966, 908, 980, 847, 922, 931, 975, 954, 771, 838, 932, 957, 966, 930, 902, 911, 939, 581]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [30]:
!pip install tiktoken

In [31]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [32]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

TypeError: expected string or buffer

In [33]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk.page_content))  # .page_content 추가

print(length)
print(tiktoken_length)

[939, 916, 994, 874, 906, 923, 951, 967, 952, 930, 962, 956, 973, 974, 858, 973, 993, 958, 978, 956, 842, 889, 942, 975, 974, 892, 977, 958, 951, 924, 975, 966, 908, 980, 847, 922, 931, 975, 954, 771, 838, 932, 957, 966, 930, 902, 911, 939, 581]
[197, 183, 204, 173, 188, 185, 195, 203, 197, 205, 214, 214, 214, 208, 181, 212, 218, 215, 211, 211, 197, 189, 214, 213, 193, 182, 215, 211, 202, 204, 214, 210, 211, 210, 164, 193, 200, 195, 190, 156, 173, 193, 188, 214, 191, 196, 197, 209, 142]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [34]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [35]:
client = openai.OpenAI()

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [37]:
import os

# API 키를 직접 입력 (따옴표 안에 본인의 실제 키 입력)
os.environ['OPENAI_API_KEY'] = "sk-..."  # 여기에 실제 키 입력

# 확인
key = os.environ.get('OPENAI_API_KEY')
if key:
    print(f"✅ 등록 완료: sk-...{key[-4:]}")

✅ 등록 완료: sk-...-...


In [38]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

NameError: name 'client' is not defined

In [40]:
import openai

client = openai.OpenAI()
print("✅ client 생성 완료")

✅ client 생성 완료


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [39]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [41]:
#!curl ipinfo.io

In [42]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [50]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [57]:
import os
from langchain_openai import OpenAIEmbeddings

os.environ['OPENAI_API_KEY'] = ""

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

embeddings = embedding_model.embed_documents([
    "This is red apple.",
    "This is yellow banana.",
    "This is green lime.",
])

print(f"✅ 성공! 벡터 차원: {len(embeddings[0])}")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [56]:
# HuggingFace 설치
!pip install -q sentence-transformers

In [58]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# 무료 embedding 모델 (API 키 불필요)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.embed_documents([
    "This is red apple.",
    "This is yellow banana.",
    "This is green lime.",
])

print(f"✅ 성공! 문서 수: {len(embeddings)}")
print(f"✅ 벡터 차원: {len(embeddings[0])}")

/tmp/ipykernel_21687/732911290.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 성공! 문서 수: 3
✅ 벡터 차원: 384


In [59]:
print(embeddings[1])

[-0.043598730117082596, 0.046768199652433395, -0.026038166135549545, -0.0013704403536394238, 0.06055562198162079, 0.05017256364226341, 0.11730732768774033, -0.027721069753170013, 0.009279526770114899, 0.07751473039388657, 0.001512883580289781, -0.09486886113882065, 0.0018310161540284753, 0.010358039289712906, 0.02260895073413849, 0.043929990381002426, -0.013756091706454754, -0.020579246804118156, -0.0327138714492321, -0.02561490423977375, 0.043547168374061584, 0.08289312571287155, -0.05014246329665184, -0.04123270511627197, -0.05022479221224785, 0.04015067592263222, 0.05747781693935394, 0.012200281023979187, 0.028081471100449562, -0.0607445202767849, -0.0662989392876625, 0.02342132292687893, 0.04859095811843872, -0.015047372318804264, -0.020352834835648537, -0.003334973705932498, 0.03370742127299309, -0.10650115460157394, 0.034283142536878586, 0.06269633769989014, 0.03697563335299492, 0.039933156222105026, 0.060280971229076385, -0.052957531064748764, -0.02507835440337658, 0.06646721810

In [60]:
len(embeddings[1])

384

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [61]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [62]:
query = ["this is red fruit"]

In [63]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7799129958370878
0.602116871556071
0.5388708787265135


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [1]:
!pip install chromadb

In [2]:
!pip install langchain-chroma

In [3]:
from langchain_chroma import Chroma

In [4]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [5]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

NameError: name 'PyPDFLoader' is not defined

In [6]:
# 필요한 모든 임포트 + PDF 로드 + 청크 분할 한 번에
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import tiktoken

# tiktoken 길이 함수
tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# PDF 로드
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load()

# 청크 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    length_function=tiktoken_len
)
docs = text_splitter.split_documents(pages)

print(f"✅ 총 청크 수: {len(docs)}")

/tmp/ipykernel_47728/1590119291.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ 총 청크 수: 182


In [7]:
# Chroma에 임베딩 저장
from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embedding_model)
print("✅ VectorStore 구축 완료")

NameError: name 'embedding_model' is not defined

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# embedding 모델 생성
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ embedding_model 준비 완료")

# Chroma VectorStore 구축
db = Chroma.from_documents(docs, embedding_model)
print(f"✅ VectorStore 구축 완료: {db._collection.count()}개 청크 저장")

/tmp/ipykernel_47728/736727281.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ embedding_model 준비 완료
✅ VectorStore 구축 완료: 182개 청크 저장


In [9]:
query = "how Demian look like?"
results = db.similarity_search(query)
print(results[0].page_content)

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how

In [ ]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [10]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [11]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [12]:
print(docs[0].page_content)

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [13]:
!pip install -U langchain langchain-classic

In [14]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [15]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

NameError: name 'ChatOpenAI' is not defined

In [16]:
!pip install -q transformers accelerate

In [19]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",          # text2text-generation → text-generation 으로 변경
    model="google/flan-t5-base",
    max_new_tokens=256,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ LLM 준비 완료")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM

✅ LLM 준비 완료


/tmp/ipykernel_47728/1387370589.py:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [22]:
from langchain.chains import RetrievalQA  # 기존
# ↓ 아래로 변경
from langchain_community.chains import RetrievalQA

ModuleNotFoundError: No module named 'langchain.chains'

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Prompt 정의
prompt = ChatPromptTemplate.from_template("""
아래 문서를 참고해서 질문에 답하세요.

문서:
{context}

질문: {question}
답변:
""")

# 검색된 문서를 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain 조립
retriever = db.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain 생성 완료")

✅ RAG Chain 생성 완료


In [24]:
# 질문하기
result = rag_chain.invoke("Who is Demian?")
print(result)

Token indices sequence length is longer than the specified maximum sequence length for this model (1280 > 512). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: 
아래 문서를 참고해서 질문에 답하세요.

문서:
DEMIAN 
character and have some significance. But I merely knew 
that Demian's mother was reported to be very wealthy. 
It was also said that neither she nor her son ever 
attended church. One boy wondered whether they might 
not be Jews but they could equally well be Mohamme­
dans. Tales were also current of Max Demian's physical 
prowess. He certainly had greatly humiliated the strong­
est boy in his class who had challenged him to a fight 
and called him a coward when he refused to fight. Those 
who were present said Demian had just taken him by the 
scruff of the neck with one hand and squeezed hard, 
whereupon the boy had gone white and crept off. He 
was unable to use his arm for days afterwards. The 
whole of one evening it was rumoured that he had died. 
For a time no assertion was too extravagant to be 
believed, everything about him was amazing and excit­
ing. Then they had enough, temporarily at any rate. Not 
much later there was further g

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [25]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [26]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_47728/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


마크다운 형식으로 출력해봅니다

In [27]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how my friend would 
look if and when I ever found him again. The woman I 
loved, if ever I had a lover, would look like that. It was 
the pattern of my life and death; it expressed the tone 
and rhythm of my fate. 
During those weeks I had embarked on some reading 
which made a deeper impression on me than anything I 
had read before. Even in later life I have seldom been 
so completely absorbed by any books, perhaps not even 
excepting Nietzsche's. It was a volume of Novalis• con-
• T,anslalor, not.. The novel D,mian was fint publiahed under the 
paeuc:IO!lym Emil Sinclair, the name or a friend or the poet Novalil 
whom Hme 10 much admired, · 
91 
Downloaded from https://www.holybooks.com

DEMIAN 
character and have some significance. But I merely knew 
that Demian's mother was reported to be very wealthy. 
It was also said that neither she nor her son ever 
attended church. One boy wondered whether they might 
not be Jews but they could equally well be Mohamme­
dans. Tales were also current of Max Demian's physical 
prowess. He certainly had greatly humiliated the strong­
est boy in his class who had challenged him to a fight 
and called him a coward when he refused to fight. Those 
who were present said Demian had just taken him by the 
scruff of the neck with one hand and squeezed hard, 
whereupon the boy had gone white and crept off. He 
was unable to use his arm for days afterwards. The 
whole of one evening it was rumoured that he had died. 
For a time no assertion was too extravagant to be 
believed, everything about him was amazing and excit­
ing. Then they had enough, temporarily at any rate. Not 
much later there was further gossip among us; some boys 
reported that Demian associated with girls and "knew 
everything.' 
Meanwhile my business with Franz Kromer followed 
its inevitable course. I could not escape him, for even 
when he left me in peace for a few days, I was still bound 
to him. He haunted my dreams like my own shadow and 
any spell that he failed to cast over me in reality my 
imagination allowed him to cast in those dreams in 
which I became utterly his slave. I lived in them-I was 
always a great dreamer-and I used up my health and 
strength more in these shadows than in real life. A 
recurrent nightmare was that Kromer was torturing me, 
spitting and kneeling on me and, what was worse, lead-
86 
Downloaded from https://www.holybooks.com

DEMIAN 
peculiar fascination for me, and I obse"ed his bright, 
clever, unusually resolute face bent diligently over his 
work; he looked less l'lte a schoolboy doing his "prep.' 
than a research student absorbed in some individual 
problem of his own. He did not attract me; I was con­
scious, on the contrary, of a certain antipathy between 
us; he was too self-possessed and cool, too defiantly con­
fident; his eyes had a grown-up expression-which never 
commends them to young people-faintly sad, with 
flashes of derision. But I could not help staring at him 
whether I liked him or not; hardly had he given me one 
glance, however, when I immediately averted my head 
in panic When I think back to it to-day and how he 
looked as a schoolboy, I can certainly confirm that he 
was different from all the rest in every respect; an indi­
vidual wholly in his own right, with his own personality 
stamped on him Therefore-though he made every effort 
not to impress-he bore himself and behaved in every 
way like a prince moving about among peasants incog­
nito and taking great pains to look like them. 
He was walking behind me on the way home from 
school. When the others had run off, he caught me up 
and greeted me. Even his style of greeting, despite the 
fact that he imitated our schoolboy tone of voice, was 
grown-up and courteous. 
'"Shall we walk. along together for a while?" he asked 
amiably. I was flattered and nodded. Then I described 
to him where I lived. 
'"Oh, there?" jie smiled. '"I know the house. An odd 
thing has been built in above your front entrance; it has 
always intrigued me.'' 
Downloaded from https://www.holybooks.com

Question: how demian looks like
Helpful Answer:

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [29]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


NameError: name 'ChatOpenAI' is not defined

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [30]:
!pip install -q langchain langchain-community langchain-core
!pip install -q langchain-text-splitters langchain-chroma
!pip install -q pypdf tiktoken sentence-transformers chromadb

Text splitter 사용을 위한 준비입니다

In [31]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    return len(tokenizer.encode(text))

### Step 1 Document loader

In [32]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load()

print(f"✅ 총 페이지 수: {len(pages)}")

✅ 총 페이지 수: 182


### Step 2 Text splitters

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len
)
docs = text_splitter.split_documents(pages)

print(f"✅ 총 청크 수: {len(docs)}")

✅ 총 청크 수: 182


### Step 3 Vector Empeddings

In [34]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding 모델 준비 완료")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding 모델 준비 완료


In [35]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)
print(f"✅ VectorStore 구축 완료: {vectorstore._collection.count()}개 청크")

✅ VectorStore 구축 완료: 546개 청크


### Step 4 Retrievers

In [36]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)
print("✅ Retriever 생성 완료")

✅ Retriever 생성 완료


In [37]:
# 검색 테스트
query = "What does Demian look like?"
retrieved_docs = retriever.invoke(query)

print(f"검색된 청크 수: {len(retrieved_docs)}")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- 결과 {i+1} (페이지 {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:200])

검색된 청크 수: 3

--- 결과 1 (페이지 89) ---
DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not 

--- 결과 2 (페이지 142) ---
VII 
Eva 
One time during the holidays I visited the house where 
-years before, Demian had lived with his mother. An old 
woman was strolling in the garden, I spoke to her and 
learned that the house

--- 결과 3 (페이지 33) ---
DEMIAN 
character and have some significance. But I merely knew 
that Demian's mother was reported to be very wealthy. 
It was also said that neither she nor her son ever 
attended church. One boy won


### Step 5 Question Answering

In [38]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
아래 문서를 참고해서 질문에 답하세요.
문서에 없는 내용은 "문서에서 찾을 수 없습니다"라고 답하세요.

문서:
{context}

질문: {question}
답변:
""")

def format_docs(docs):
    return "\n\n".join(
        f"[페이지 {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    )

from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    do_sample=False
)
llm = HuggingFacePipeline(pipeline=pipe)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain 완성!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

✅ RAG Chain 완성!


In [39]:
# 최종 질문
questions = [
    "What does Demian look like?",
    "Who is Emil Sinclair?",
    "What is the relationship between Sinclair and Demian?"
]

for q in questions:
    print(f"\n❓ {q}")
    print(f"💬 {rag_chain.invoke(q)}")
    print("-" * 50)

Token indices sequence length is longer than the specified maximum sequence length for this model (1236 > 512). Running this sequence through the model will result in indexing errors



❓ What does Demian look like?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 Human: 
아래 문서를 참고해서 질문에 답하세요.
문서에 없는 내용은 "문서에서 찾을 수 없습니다"라고 답하세요.

문서:
[페이지 89]
DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face so

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 Human: 
아래 문서를 참고해서 질문에 답하세요.
문서에 없는 내용은 "문서에서 찾을 수 없습니다"라고 답하세요.

문서:
[페이지 122]
JA~OB AND THE ANGEL 
exhortation to Demian's words, which I had been carry­
ing round with me for years. They knew nothing of 
each other and yet both had given me the same message. 
"The things we see," said Pistorius gently, "are the 
thipgs which are already in us. There is no reality 
beyond what we have inside us. That is why most people 
live such unreal lives; they take pictures outside them­
selves for the real ones and fail to express their own 
world. One can of course live contentedly enough in 
that situation. But once you know about the other you 
no longer have the choice of following the majority 
way. The way of the majority, Sinclair, is easy, ours is 
hard ... But now we must go." 
Some days later after twice waiting for him in vain 
he came swaying along round a street-corner in the cold 
night wind; he was reeling along, quite drunk. I felt 
no wish to call after him. He went past me 

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 Human: 
아래 문서를 참고해서 질문에 답하세요.
문서에 없는 내용은 "문서에서 찾을 수 없습니다"라고 답하세요.

문서:
[페이지 122]
JA~OB AND THE ANGEL 
exhortation to Demian's words, which I had been carry­
ing round with me for years. They knew nothing of 
each other and yet both had given me the same message. 
"The things we see," said Pistorius gently, "are the 
thipgs which are already in us. There is no reality 
beyond what we have inside us. That is why most people 
live such unreal lives; they take pictures outside them­
selves for the real ones and fail to express their own 
world. One can of course live contentedly enough in 
that situation. But once you know about the other you 
no longer have the choice of following the majority 
way. The way of the majority, Sinclair, is easy, ours is 
hard ... But now we must go." 
Some days later after twice waiting for him in vain 
he came swaying along round a street-corner in the cold 
night wind; he was reeling along, quite drunk. I felt 
no wish to call after him. He went past me 

In [40]:
!wget -q "https://www.holybooks.com/wp-content/uploads/Demian-By-Hermann-Hesse.pdf" -O /content/Demian.pdf

In [49]:
import os
# /content 폴더의 파일 목록 확인
for f in os.listdir("/content"):
    if f.endswith(".ipynb"):
        print(f)

In [52]:
import os

# Drive에서 찾기
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if f.endswith(".ipynb") and "RAG" in f:
            print(os.path.join(root, f))

/content/drive/MyDrive/Colab Notebooks/Day1_RAG_Code.ipynb
/content/drive/MyDrive/Colab Notebooks/Day1_RAG.ipynb


In [53]:
import os
for f in os.listdir("/content"):
    if f.endswith(".ipynb"):
        print(f)

In [54]:
import json

filename = "/content/drive/MyDrive/Colab Notebooks/Day1_RAG_Code.ipynb"

with open(filename, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]
    print("✅ widgets 제거 완료")
else:
    print("widgets 없음")

with open(filename, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("✅ 저장 완료")

✅ widgets 제거 완료
✅ 저장 완료


# 🗓️ RAG 시작하기 — 실습 회고

## 오늘 한 것
LangChain의 핵심 컴포넌트를 하나씩 실습하며 RAG 파이프라인을 처음부터 끝까지 직접 구현했다.

- **Document Loader** — PDF, CSV, 웹페이지를 Document 객체로 불러오기
- **Text Splitter** — 긴 문서를 토큰 기준으로 청크 분할
- **Embeddings** — 텍스트를 의미 벡터로 변환 (HuggingFace 무료 모델 활용)
- **VectorStore** — Chroma로 벡터 저장 및 유사도 검색
- **Retriever** — MMR 방식으로 다양한 문서 검색
- **RAG Chain** — 검색 + 생성 파이프라인 완성

---

## 잘 된 것 👍
- 오류가 반복되어도 포기하지 않고 끝까지 완주했다
- `wget`으로 파일을 직접 다운로드하는 방법을 익혔다
- OpenAI 크레딧 없이도 HuggingFace 무료 모델로 전체 파이프라인을 완성했다
- `"how demian looks like?"` 쿼리로 실제 관련 문서가 검색되는 것을 눈으로 확인했다

---

## 어려웠던 것 😅
- 런타임 재시작 후 환경 변수, 파일, 모델이 모두 초기화되는 것을 여러 번 겪었다
- Colab Secrets의 토글과 이름이 코드와 일치해야 한다는 것을 몰라서 시간이 걸렸다
- `langchain-community` deprecated 경고와 `pydantic` 버전 충돌 오류가 혼란스러웠다
- `files.upload()`가 세션 재시작 후 동작하지 않는 문제로 한 시간 이상 소요됐다

---

## 배운 것 💡

### 기술적 배움
- LangChain 셀은 반드시 **위에서 아래로 순서대로** 실행해야 한다
- `load_and_split()` 대신 `load()` + 별도 Splitter 사용이 더 안정적이다
- `CharacterTextSplitter`는 구분자가 없으면 분할이 안 되므로 `RecursiveCharacterTextSplitter`가 더 범용적이다
- Chroma의 유사도 점수는 **0에 가까울수록 유사**하다 (거리 기반)
- MMR은 `fetch_k > k` 로 설정해 후보를 넉넉히 뽑은 뒤 다양성 기준으로 재정렬한다

### RAG에 대한 이해
- RAG의 가장 큰 장점은 **답변의 출처를 문서에서 직접 확인**할 수 있다는 것이다
- Fine-tuning은 "기억을 늘리는 것", RAG는 "필요할 때 찾아보는 것"이다
- 모델 크기보다 **검색 품질**이 최종 답변 품질을 결정한다

---

## 다음에 해볼 것 🚀
- [ ] Advanced RAG (HyDE, Parent-Document Retrieval, Re-ranking)
- [ ] LangSmith로 파이프라인 디버깅
- [ ] 한국어 문서에 적합한 Embedding 모델 탐색
- [ ] AI Agent 패러다임으로 확장